# Experiment Result Reproduction

This notebook regenerates the four figures in `assets/result-*` from benchmark CSV files.
It replaces the old exploratory cells with a reproducible workflow:

1. Run benchmark scripts to create CSV files under `results/2d` and `results/3d`.
2. Read those CSV files.
3. Save the result figures back into `assets/`.

## How to Run the Experiments

Run these commands from the repository root.

### 2D results

```sh
conda activate r2r-handover
python experiments/experiment_2d.py --variants rrt connect bidirectional star prm --output_dir results/2d
```

For a quick smoke test, add `--sample_counts 100` to the command.

This creates:

```text
results/2d/results_rrt.csv
results/2d/results_connect.csv
results/2d/results_bi.csv
results/2d/results_star.csv
results/2d/results_prm.csv
```

### 3D CoppeliaSim results

Start CoppeliaSim, open `envs/R2R.ttt`, run the scene, then execute:

```sh
conda activate r2r-handover
python experiments/experiment.py --variants rrt connect bidirectional star prm --output_dir results/3d --trace_star --trace_samples 1000
```

For a quick CoppeliaSim smoke test, add `--sample_counts 100 --trace_samples 100`.

This creates the same planner comparison CSV files under `results/3d`, plus:

```text
results/3d/results_star_r1.csv
results/3d/results_star_r2.csv
```

Those trace files are used for `result-time component.png` and `result-test&dist.png`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
ASSETS = ROOT / "assets"
RESULTS_2D = ROOT / "results" / "2d"
RESULTS_3D = ROOT / "results" / "3d"

ASSETS.mkdir(exist_ok=True)

PLANNER_FILES = {
    "RRT-connect": "results_connect.csv",
    "RRT-bidirectional": "results_bi.csv",
    "PRM": "results_prm.csv",
    "RRT": "results_rrt.csv",
    "RRT*": "results_star.csv",
}

## Optional: Run Benchmarks From the Notebook

The 2D benchmark is safe to run from here. The 3D benchmark requires CoppeliaSim to be running.
Set the flags to `True` only when you want the notebook to regenerate CSV files.

In [ ]:
RUN_2D = False
RUN_3D = False
RUN_3D_TRACE = False

if RUN_2D:
    subprocess.run(
        [
            sys.executable,
            "experiments/experiment_2d.py",
            "--variants",
            "rrt",
            "connect",
            "bidirectional",
            "star",
            "prm",
            "--output_dir",
            str(RESULTS_2D),
        ],
        check=True,
    )

if RUN_3D:
    command = [
        sys.executable,
        "experiments/experiment.py",
        "--variants",
        "rrt",
        "connect",
        "bidirectional",
        "star",
        "prm",
        "--output_dir",
        str(RESULTS_3D),
    ]
    if RUN_3D_TRACE:
        command.extend(["--trace_star", "--trace_samples", "1000"])
    subprocess.run(command, check=True)

In [ ]:
def require_files(paths, run_hint):
    missing = [path for path in paths if not path.exists()]
    if missing:
        missing_text = "\n".join(str(path) for path in missing)
        raise FileNotFoundError(f"Missing required CSV files:\n{missing_text}\n\nRun:\n{run_hint}")


def load_planner_results(results_dir):
    paths = {label: Path(results_dir) / filename for label, filename in PLANNER_FILES.items()}
    require_files(
        paths.values(),
        f"python experiments/experiment_2d.py --variants rrt connect bidirectional star prm --output_dir {results_dir}",
    )
    return {label: pd.read_csv(path) for label, path in paths.items()}


def total_time(label, df):
    if label == "PRM" and {"r1_build", "r2_build"}.issubset(df.columns):
        return df["r1_build"] + df["r2_build"]
    return df["r1_total"] + df["r2_total"]


def plot_planner_comparison(results_dir, output_path):
    data = load_planner_results(results_dir)
    sample_values = next(iter(data.values()))["num_samples"].tolist()
    split_index = sum(value < 1000 for value in sample_values)
    x1 = sample_values[:split_index]
    x2 = sample_values[split_index:]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    for label, df in data.items():
        y = total_time(label, df)
        ax1.plot(x1, y.iloc[:split_index], label=label)
        ax2.plot(x2, y.iloc[split_index:], label=label)

    ax1.set_title("Sample (100–900)")
    ax1.set_xlabel("Number of Samples")
    ax1.set_ylabel("Time (s)")
    ax1.grid(True)
    ax1.legend()

    ax2.set_title("Sample (1000–5000)")
    ax2.set_xlabel("Number of Samples")
    ax2.grid(True)
    ax2.legend()

    fig.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=100)
    plt.show()
    return output_path

In [ ]:
def load_star_trace(results_dir):
    path1 = Path(results_dir) / "results_star_r1.csv"
    path2 = Path(results_dir) / "results_star_r2.csv"
    require_files(
        [path1, path2],
        f"python experiments/experiment.py --output_dir {results_dir} --trace_star --trace_samples 1000",
    )
    return pd.read_csv(path1), pd.read_csv(path2)


def plot_component_time(results_dir, output_path):
    data1, data2 = load_star_trace(results_dir)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 7), sharex=True)
    fig.suptitle("Component Time Breakdown per Iteration")

    ax1.plot(data1["iter"], data1["r1_total"], label="Total Time (R1)", color="blue")
    ax1.plot(data1["iter"], data1["r1_collision"], label="Collision Time (R1)", color="c", linestyle="dashed")
    ax1.plot(data1["iter"], data1["r1_near"], label="Nearest Time (R1)", color="blue", linestyle="dotted")
    ax1.plot(data1["iter"], data1["r1_neighbors"], label="Neighbor Time (R1)", color="c", linestyle="dashdot")
    ax1.set_ylabel("Time (s)")
    ax1.legend(loc="upper right")
    ax1.grid(True)

    ax2.plot(data2["iter"], data2["r2_total"], label="Total Time (R2)", color="red")
    ax2.plot(data2["iter"], data2["r2_collision"], label="Collision Time (R2)", color="orange", linestyle="dashed")
    ax2.plot(data2["iter"], data2["r2_near"], label="Nearest Time (R2)", color="red", linestyle="dotted")
    ax2.plot(data2["iter"], data2["r2_neighbors"], label="Neighbor Time (R2)", color="orange", linestyle="dashdot")
    ax2.set_xlabel("Iteration")
    ax2.set_ylabel("Time (s)")
    ax2.legend(loc="upper right")
    ax2.grid(True)

    fig.tight_layout(rect=[0, 0.03, 1, 0.95])
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=100)
    plt.show()
    return output_path


def plot_time_vs_path(results_dir, output_path):
    data1, data2 = load_star_trace(results_dir)

    fig, ax1 = plt.subplots(figsize=(7, 5))
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel("Total Time")
    ax1.plot(data1["iter"], data1["r1_total"], color="tab:blue", label="Total Time (R1)")
    ax1.plot(data2["iter"], data2["r2_total"], color="tab:green", label="Total Time (R2)")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.set_ylabel("Path Length")
    ax2.plot(data1["iter"], data1["r1_path"], color="tab:red", linestyle="--", label="Path Length (R1)")
    ax2.plot(data2["iter"], data2["r2_path"], color="tab:orange", linestyle="--", label="Path Length (R2)")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    fig.legend()
    plt.title("Total Time vs Path Length")
    fig.tight_layout()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=100)
    plt.show()
    return output_path

## Generate Figures

Run the cells below after the corresponding CSV files exist.

In [ ]:
plot_planner_comparison(RESULTS_2D, ASSETS / "result-2d.png")

In [ ]:
plot_planner_comparison(RESULTS_3D, ASSETS / "result-3d.png")

In [ ]:
plot_component_time(RESULTS_3D, ASSETS / "result-time component.png")

In [ ]:
plot_time_vs_path(RESULTS_3D, ASSETS / "result-test&dist.png")